# [Project title]

**Team members:** [Names]  
**Repository:** [URL of your team's fork]

## Setup

Use relative paths and keep the notebook runnable from beginning to end.

In [3]:
import pandas as pd
import requests

CENSUS_API_KEY = "9a206f12f4116b743df3da5650ee1dc0a71f363c"
YEAR = "2024"
BASE_URL = f"https://api.census.gov/data/{YEAR}/acs/acs5/profile"

# DP03_0062E: Median household income
# DP03_0128PE: Poverty rate (%)
# DP02_0068PE: Educational attainment (% Bachelor's or higher)
# DP03_0096PE: Health insurance coverage rate (%)
variables = [
    "NAME",
    "DP03_0062E",
    "DP03_0128PE",
    "DP02_0068PE",
    "DP03_0096PE",
]
params = {
    "get": ",".join(variables),
    "for": "county:*",
    "in": "state:*",
    "key": CENSUS_API_KEY,
}

response = requests.get(BASE_URL, params=params)

if response.status_code != 200:
    raise RuntimeError(
        f"Census API request failed ({response.status_code}): {response.text}"
    )

raw_json = response.json()
headers = raw_json[0]
rows = raw_json[1:]

df_raw = pd.DataFrame(rows, columns=headers)

output_filename = "census_county_data_raw.csv"
df_raw.to_csv(output_filename, index=False)

In [ ]:
from pathlib import Path

DATA_DIR = Path("data")
assert DATA_DIR.is_dir(), "The data directory is missing."

## Load data

In [ ]:
# Census ACS 2024 5-year county profile (pulled by the API cell above)
census_raw = pd.read_csv(
    DATA_DIR / "census_county_data_raw.csv",
    dtype={"state": str, "county": str},
)

# CDC Diabetes Atlas: 2 title lines above the header and a 1-line citation footer
diabetes_raw = pd.read_csv(
    DATA_DIR / "DiabetesAtlas_CountyData.csv",
    skiprows=2,
    skipfooter=1,
    engine="python",
    dtype={"CountyFIPS": str},
)

print("Census:", census_raw.shape)
print("Diabetes:", diabetes_raw.shape)

## Prepare data

In [ ]:
import numpy as np

# --- Census ---
census = census_raw.rename(columns={
    "DP03_0062E": "median_household_income",
    "DP03_0128PE": "poverty_rate",
    "DP02_0068PE": "pct_bachelors_or_higher",
    "DP03_0096PE": "pct_health_insured",
})
census["fips"] = census["state"].str.zfill(2) + census["county"].str.zfill(3)
census[["county_name", "state_name"]] = census["NAME"].str.rsplit(", ", n=1, expand=True)

census_value_cols = [
    "median_household_income",
    "poverty_rate",
    "pct_bachelors_or_higher",
    "pct_health_insured",
]
# The Census API uses large negative sentinels (e.g. -666666666) for
# estimates that could not be computed; treat them as missing
census[census_value_cols] = census[census_value_cols].mask(census[census_value_cols] < 0)

# Puerto Rico (state 72) is not covered by the Diabetes Atlas
census = census[census["state"] != "72"]
census = census[["fips", "county_name", "state_name"] + census_value_cols]

# --- Diabetes ---
diabetes = diabetes_raw.rename(columns=lambda c: c.strip()).rename(columns={
    "County": "county_name",
    "State": "state_name",
    "CountyFIPS": "fips",
    "Percentage": "diabetes_pct",
    "Lower Limit": "diabetes_pct_lower",
    "Upper Limit": "diabetes_pct_upper",
})
diabetes["fips"] = diabetes["fips"].str.zfill(5)
diabetes_value_cols = ["diabetes_pct", "diabetes_pct_lower", "diabetes_pct_upper"]
diabetes[diabetes_value_cols] = diabetes[diabetes_value_cols].apply(
    pd.to_numeric, errors="coerce"  # "No Data" -> NaN
)

assert census["fips"].is_unique and diabetes["fips"].is_unique

# --- Merge on county FIPS ---
merged = diabetes.drop(columns=["county_name", "state_name"]).merge(
    census, on="fips", how="inner", validate="one_to_one"
)
merged = merged[["fips", "county_name", "state_name"] + census_value_cols + diabetes_value_cols]
print("Counties after merge:", len(merged))

# Every Tennessee county is "No Data" in the 2024 Atlas, so they have no outcome
no_outcome = merged["diabetes_pct"].isna()
print("Dropping counties with no diabetes estimate:")
print(merged.loc[no_outcome, "state_name"].value_counts().to_string())
merged = merged[~no_outcome].reset_index(drop=True)

print("Final analytic dataset:", merged.shape)
print("Remaining missing values:")
print(merged.isna().sum()[lambda s: s > 0].to_string())

census.to_csv(DATA_DIR / "census_county_clean.csv", index=False)
diabetes.to_csv(DATA_DIR / "diabetes_county_clean.csv", index=False)
merged.to_csv(DATA_DIR / "county_merged.csv", index=False)

## Explore data

In [ ]:
# Summarize and visualize the prepared data.

## Build models

In [ ]:
# Train baseline and candidate models.

## Evaluate models

In [ ]:
# Compare models using the chosen evaluation measures.

## Final results

In [ ]:
# Generate the final tables and figures.

## AI usage disclosure

- **AI tool/model:** [Name the tool and model, or state that no generative AI was used.]
- **Used for:** [Briefly identify the tasks.]
- **How we verified the work:** [Briefly identify the checks.]
- **Prompt(s) used:** [Paste the prompts used.]